# Stage 3 — BLAIR Embeddings
Encodes all 137,269 products using `hyp1231/blair-roberta-large`.

**Steps:**
1. Check GPU
2. Install dependencies
3. Upload `products_rich.parquet`
4. Embed all products
5. Verify embeddings
6. Download `item_embeddings.npy` + `item_ids.npy`

**Runtime:** ~25–35 min on T4 GPU for 137k items.

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Embedding will be very slow on CPU.')

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────
!pip install transformers torch pandas numpy pyarrow tqdm --quiet
print('Dependencies installed.')

In [ ]:
# ── Cell 3: Upload products_rich.parquet ───────────────────────────────────
from google.colab import files
print('Upload products_rich.parquet from data/processed/ ...')
uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))

In [ ]:
# ── Cell 4: Embed all products ─────────────────────────────────────────────
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.notebook import tqdm
from transformers import AutoModel, AutoTokenizer

# ── Config (matches configs/config.yaml stage3) ────────────────────────────
MODEL_NAME       = 'hyp1231/blair-roberta-large'
BATCH_SIZE       = 32       # safe for T4 + RoBERTa-large
MAX_LENGTH       = 512
EMB_DIM          = 1024
CHECKPOINT_EVERY = 5000
PARQUET_PATH     = 'products_rich.parquet'
OUT_EMB          = 'item_embeddings.npy'
OUT_IDS          = 'item_ids.npy'
PARTIAL_EMB      = 'item_embeddings_partial.npy'
LAST_IDX_FILE    = 'last_index.txt'

# ── Load data ──────────────────────────────────────────────────────────────
df = pd.read_parquet(PARQUET_PATH)
texts  = df['rich_text'].fillna('').tolist()
asins  = df['parent_asin'].tolist()
n_items = len(texts)
print(f'Loaded {n_items:,} products')

# ── Device ─────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Load model ─────────────────────────────────────────────────────────────
print(f'Loading {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
print('Model ready.')

# ── Checkpoint resume ──────────────────────────────────────────────────────
start_idx = 0
embeddings = np.zeros((n_items, EMB_DIM), dtype=np.float32)
if Path(PARTIAL_EMB).exists() and Path(LAST_IDX_FILE).exists():
    start_idx  = int(Path(LAST_IDX_FILE).read_text().strip())
    partial    = np.load(PARTIAL_EMB)
    if partial.shape == (start_idx, EMB_DIM):
        embeddings[:start_idx] = partial
        print(f'Resuming from checkpoint: {start_idx}/{n_items} done')
    else:
        start_idx = 0
        print('Checkpoint shape mismatch — starting from scratch')

# ── Embedding loop ─────────────────────────────────────────────────────────
t0 = time.time()
batch_starts = range(start_idx, n_items, BATCH_SIZE)

for batch_start in tqdm(batch_starts, desc='Embedding', unit='batch'):
    batch_end   = min(batch_start + BATCH_SIZE, n_items)
    batch_texts = texts[batch_start:batch_end]

    enc = tokenizer(
        batch_texts,
        max_length=MAX_LENGTH,
        padding=True,
        truncation=True,
        return_tensors='pt',
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**enc)

    cls = out.last_hidden_state[:, 0, :]
    cls = cls / cls.norm(dim=1, keepdim=True)
    embeddings[batch_start:batch_end] = cls.cpu().float().numpy()

    items_done = batch_end
    if items_done % CHECKPOINT_EVERY < BATCH_SIZE:
        np.save(PARTIAL_EMB, embeddings[:items_done])
        Path(LAST_IDX_FILE).write_text(str(items_done))
        elapsed = time.time() - t0
        print(f'Checkpoint: {items_done}/{n_items}  ({items_done/elapsed:.1f} items/s)')

elapsed = time.time() - t0
print(f'\nEmbedding complete: {n_items:,} items in {elapsed/60:.1f} min  ({n_items/elapsed:.1f} items/s)')

# ── Save ───────────────────────────────────────────────────────────────────
np.save(OUT_EMB, embeddings)
np.save(OUT_IDS, np.array(asins, dtype=object))
# Clean up checkpoints
for f in [PARTIAL_EMB, LAST_IDX_FILE]:
    if Path(f).exists(): Path(f).unlink()
print(f'Saved {OUT_EMB}  shape={embeddings.shape}')
print(f'Saved {OUT_IDS}  shape={len(asins)}')

In [ ]:
# ── Cell 5: Verify embeddings ──────────────────────────────────────────────
import numpy as np

emb  = np.load('item_embeddings.npy')
ids  = np.load('item_ids.npy', allow_pickle=True)

print('=== Shape verification ===')
print(f'item_embeddings.npy : {emb.shape}  dtype={emb.dtype}')
print(f'item_ids.npy        : {ids.shape}  dtype={ids.dtype}')
assert emb.shape[1] == 1024, f'Expected dim=1024, got {emb.shape[1]}'
assert emb.shape[0] == ids.shape[0], 'Mismatch between embedding count and id count'

print('\n=== L2 norm verification ===')
norms = np.linalg.norm(emb, axis=1)
print(f'Norm min={norms.min():.6f}  max={norms.max():.6f}  mean={norms.mean():.6f}')
assert np.allclose(norms, 1.0, atol=1e-5), 'Embeddings are not unit-normalised!'
print('All embeddings are L2-normalised. OK')

print('\n=== Random pair cosine similarities ===')
rng = np.random.default_rng(42)
idx = rng.choice(len(emb), size=6, replace=False)
for i in range(3):
    a, b = idx[i*2], idx[i*2+1]
    sim  = float(np.dot(emb[a], emb[b]))
    print(f'  {ids[a][:12]:15s} <-> {ids[b][:12]:15s}  cosine={sim:.4f}')

print('\nVerification complete. Ready to download.')

In [ ]:
# ── Cell 6: Download embeddings ────────────────────────────────────────────
from google.colab import files
print('Downloading item_embeddings.npy  (~500 MB) ...')
files.download('item_embeddings.npy')
print('Downloading item_ids.npy ...')
files.download('item_ids.npy')
print('Done. Place both files in data/embeddings/ before running Stage 4.')